# Makemore Part 3 — Character-Level MLP

A from-scratch character-level language model built with PyTorch. This notebook implements embeddings, a deep MLP, manual BatchNorm, training diagnostics, evaluation, and autoregressive text generation.

## 1. Imports and setup

In [ ]:
import random
from pathlib import Path

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


## 2. Load the dataset

In [ ]:
data_candidates = [Path('data/names.txt'), Path('../data/names.txt'), Path('names.txt')]
DATA_PATH = next((p for p in data_candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        'names.txt was not found. Run scripts/download_data.py first.'
    )

words = DATA_PATH.read_text(encoding='utf-8').splitlines()
print(f'Loaded {len(words):,} names from {DATA_PATH}')
words[:8]


## 3. Character vocabulary and dataset construction

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

block_size = 3

def build_dataset(words):
    X, Y = [], []
    for word in words:
        context = [0] * block_size
        for ch in word + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

print(f'Vocabulary size: {vocab_size}')
print(f'Train: {Xtr.shape} | Validation: {Xdev.shape} | Test: {Xte.shape}')


## 4. Model components

The network uses learned character embeddings followed by several manually implemented Linear → BatchNorm → Tanh blocks. Implementing these layers explicitly makes the mechanics of the network visible instead of hiding them behind `torch.nn` modules.

In [ ]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)
            xvar = x.var(0, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []


## 5. Build the MLP

In [ ]:
n_embd = 10
n_hidden = 100
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)
layers = [
    Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]

with torch.no_grad():
    layers[-1].gamma *= 0.1

parameters = [C] + [p for layer in layers for p in layer.parameters()]
for p in parameters:
    p.requires_grad = True

print(f'Total parameters: {sum(p.nelement() for p in parameters):,}')


## 6. Train the model

In [ ]:
max_steps = 200_000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    emb = C[Xb]
    x = emb.view(emb.shape[0], -1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x, Yb)

    for layer in layers:
        if hasattr(layer, 'out') and layer.out.requires_grad:
            layer.out.retain_grad()
    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 150_000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 10_000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    with torch.no_grad():
        ud.append([((lr * p.grad).std() / p.data.std()).log10().item() for p in parameters])


## 7. Training diagnostics

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(lossi)
plt.title('Training loss (log10)')
plt.xlabel('Training step')
plt.ylabel('log10(loss)')
plt.show()

plt.figure(figsize=(12, 4))
legends = []
for i, layer in enumerate(layers[:-1]):
    if isinstance(layer, Tanh):
        t = layer.out
        print(f'layer {i} ({layer.__class__.__name__}): mean {t.mean():+.2f}, std {t.std():.2f}, saturated {(t.abs() > 0.97).float().mean()*100:.2f}%')
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legends.append(f'layer {i}')
plt.legend(legends)
plt.title('Activation distributions')
plt.show()


In [ ]:
plt.figure(figsize=(12, 4))
legends = []
for i, layer in enumerate(layers[:-1]):
    if isinstance(layer, Tanh) and layer.out.grad is not None:
        t = layer.out.grad
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legends.append(f'layer {i}')
plt.legend(legends)
plt.title('Gradient distributions')
plt.show()


In [ ]:
plt.figure(figsize=(12, 4))
legends = []
for i, p in enumerate(parameters):
    if p.ndim == 2 and p.grad is not None:
        t = p.grad
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legends.append(f'{i} {tuple(p.shape)}')
        print(f'weight {tuple(p.shape)} | grad:data ratio {t.std() / p.std():.3e}')
plt.legend(legends)
plt.title('Weight gradient distributions')
plt.show()


In [ ]:
plt.figure(figsize=(12, 4))
legends = []
for i, p in enumerate(parameters):
    if p.ndim == 2:
        plt.plot([ud[j][i] for j in range(len(ud))])
        legends.append(f'param {i}')
plt.plot([0, len(ud)], [-3, -3], 'k')
plt.legend(legends)
plt.title('Parameter update / parameter scale (log10)')
plt.show()


## 8. Evaluate on train / validation / test splits

In [ ]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]

    emb = C[x]
    x = emb.view(emb.shape[0], -1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x, y)
    print(f'{split:>5}: {loss.item():.4f}')
    return loss.item()

for layer in layers:
    layer.training = False

split_loss('train')
split_loss('val')
split_loss('test')


## 9. Generate new names

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    out = []
    context = [0] * block_size

    while True:
        emb = C[torch.tensor([context])]
        x = emb.view(emb.shape[0], -1)
        for layer in layers:
            x = layer(x)
        probs = F.softmax(x, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(''.join(itos[i] for i in out))
